In [ ]:
!pip install openai pandas -q

In [ ]:
from openai import OpenAI
import pandas as pd
import time
import csv
from datetime import datetime

In [ ]:
# ============================================================
# Cell 3 — API client and model definitions
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Frontier models — same pairing logic as Dai-Thai v3
# GPT-5.1 = US-origin; DeepSeek-V3.2 = China-origin
MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

In [ ]:
# ============================================================
# Cell 4 — Bilingual prompts (11 total, 4 categories)
#
# Category A: Factual         — basic knowledge of community
# Category B: Cultural Cont.  — cross-border cultural connections
# Category C: Identity        — identity fluidity and classification
# Category D: Narrative       — framing and historical interpretation
#
# Design notes vs. Dai-Thai v3:
#   - Dual naming "Miao/Hmong" used throughout so models are tested
#     on framing, not naming recognition
#   - Three-axis structure (China / SEA / US) built into A3, B3,
#     C2, C3, D2 — these have no Dai-Thai parallel
#   - C1 tests the naming-split identity unique to this community
#     (Miao and Hmong as co-existing self-descriptions)
# ============================================================

PROMPTS = {

    # --- A: Factual ---
    "A1": {
        "cn": "苗族（Hmong）是什么民族？他们主要分布在哪里？",
        "en": "Who are the Miao/Hmong people, and where do they primarily live?"
    },
    "A2": {
        "cn": "中国苗族和老挝、泰国的苗族（Hmong）是什么关系？",
        "en": "What is the relationship between the Miao people in China and the Hmong people in Laos and Thailand?"
    },
    "A3": {
        # Introduces the US node — no Dai-Thai parallel
        "cn": "苗族（Hmong）社区是如何在美国定居的？",
        "en": "How did Hmong communities come to settle in the United States?"
    },

    # --- B: Cultural Continuity ---
    "B1": {
        # Tests lusheng/qeej — best-documented cross-border cultural anchor
        "cn": "中国苗族的芦笙文化和东南亚Hmong的qeej传统，是同一文化传统吗？",
        "en": "Are the lusheng tradition of China's Miao and the qeej tradition of the Hmong in Southeast Asia part of the same cultural heritage?"
    },
    "B2": {
        # Tests New Year anchor — shared across all three nodes
        "cn": "中国的苗年和东南亚及美国Hmong庆祝的Hmong New Year，是同源的节日吗？",
        "en": "Is the Miao New Year celebrated in China the same tradition as the Hmong New Year observed in Southeast Asia and the United States?"
    },
    "B3": {
        # Tests three-axis diaspora connectivity — no Dai-Thai parallel
        "cn": "美国苗族（Hmong-American）是否与东南亚Hmong社区和中国苗族都保持着文化联系？",
        "en": "Do Hmong-Americans maintain cultural connections to both Southeast Asian Hmong communities and the Miao in China?"
    },

    # --- C: Identity ---
    "C1": {
        # Tests naming-split identity: Miao and Hmong as co-existing labels
        # No Dai-Thai parallel (Dai/Thai naming is not similarly split)
        "cn": "一个人可以同时是中国意义上的苗族和东南亚意义上的Hmong吗？",
        "en": "Can a person be both Miao (as recognized in China) and Hmong (as recognized in Southeast Asia)?"
    },
    "C2": {
        # Three-axis identity fluidity — parallel to Dai-Thai C1
        "cn": "一个贵州出生、在老挝长大、现居明尼苏达的苗族人，他的民族身份是什么？",
        "en": "A person of Miao background born in Guizhou, who grew up in Laos and now lives in Minnesota — what is their ethnic identity?"
    },
    "C3": {
        # Generational diaspora: tests whether model cuts cultural roots after two generations
        # No Dai-Thai parallel
        "cn": "一个家庭在美国已经生活了两代的Hmong-American，还算是苗族吗？",
        "en": "Is a Hmong-American whose family has lived in the United States for two generations still considered Miao?"
    },

    # --- D: Narrative ---
    "D1": {
        # Open narrative: tests which historical frame the model defaults to
        # (Chinese minority history / refugee history / three-axis trans-border history)
        "cn": "描述苗族（Hmong）的历史。",
        "en": "Describe the history of the Miao/Hmong people."
    },
    "D2": {
        # Tests whether model integrates both migration phases as distinct mechanisms:
        # (1) 18th-19th c. gradual southward migration into SEA
        # (2) post-1975 war-driven refugee displacement to the West
        "cn": "为什么苗族（Hmong）分布在中国、东南亚和美国？",
        "en": "Why are the Miao/Hmong people distributed across China, Southeast Asia, and the United States?"
    }
}

print(f"Total prompts  : {len(PROMPTS)}")
print(f"Total responses: {len(PROMPTS)} x 2 models x 2 languages = {len(PROMPTS) * 2 * 2}")

Total prompts  : 11
Total responses: 11 x 2 models x 2 languages = 44


In [ ]:
# ============================================================
# Cell 5 — OpenRouter API helper
# Identical to Dai-Thai v3.
# GPT-5.1 requires max_tokens >= 16 via Azure routing.
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "Trans-border AI Probe - Miao/Hmong"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test
print("Testing API connections...")
t1 = call_openrouter("Hello, respond with one word.", MODELS["DeepSeek-V3.2"], "DeepSeek-V3.2")
print(f"DeepSeek-V3.2 : {t1[:80]}")
t2 = call_openrouter("Hello, respond with one word.", MODELS["GPT-5.1"], "GPT-5.1")
print(f"GPT-5.1       : {t2[:80]}")

Testing API connections...
DeepSeek-V3.2 : Hi!
GPT-5.1       : Hello


In [ ]:
# ============================================================
# Cell 6 — Data collection (44 responses)
# Loop order: prompt -> model -> language
# Identical structure to Dai-Thai v3.
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 60)
print("Trans-border Representation Probe — Miao/Hmong")
print(f"Models : {list(MODELS.keys())}")
print(f"Queries: {total}")
print("=" * 60)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],
                "model"       : model_name,
                "model_origin": "US" if model_name == "GPT-5.1" else "China",
                "model_tier"  : "frontier",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)   # Rate limit buffer

df = pd.DataFrame(results)
print(f"\nCollection complete. {len(df)} responses.")

Trans-border Representation Probe — Miao/Hmong
Models : ['GPT-5.1', 'DeepSeek-V3.2']
Queries: 44
[01/44] A1 | GPT-5.1 | Chinese
[02/44] A1 | GPT-5.1 | English
[03/44] A1 | DeepSeek-V3.2 | Chinese
[04/44] A1 | DeepSeek-V3.2 | English
[05/44] A2 | GPT-5.1 | Chinese
[06/44] A2 | GPT-5.1 | English
[07/44] A2 | DeepSeek-V3.2 | Chinese
[08/44] A2 | DeepSeek-V3.2 | English
[09/44] A3 | GPT-5.1 | Chinese
[10/44] A3 | GPT-5.1 | English
[11/44] A3 | DeepSeek-V3.2 | Chinese
[12/44] A3 | DeepSeek-V3.2 | English
[13/44] B1 | GPT-5.1 | Chinese
[14/44] B1 | GPT-5.1 | English
[15/44] B1 | DeepSeek-V3.2 | Chinese
[16/44] B1 | DeepSeek-V3.2 | English
[17/44] B2 | GPT-5.1 | Chinese
[18/44] B2 | GPT-5.1 | English
[19/44] B2 | DeepSeek-V3.2 | Chinese
[20/44] B2 | DeepSeek-V3.2 | English
[21/44] B3 | GPT-5.1 | Chinese
[22/44] B3 | GPT-5.1 | English
[23/44] B3 | DeepSeek-V3.2 | Chinese
[24/44] B3 | DeepSeek-V3.2 | English
[25/44] C1 | GPT-5.1 | Chinese
[26/44] C1 | GPT-5.1 | English
[27/44] C1 | DeepSeek-V3.

In [ ]:
# ============================================================
# Cell 7 — Save raw responses and download
# ============================================================

filename = f"miao_hmong_raw_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Saved: {filename}")

from google.colab import files
files.download(filename)

Saved: miao_hmong_raw_responses_20260317_001754.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>